# Citation Retrieval, our research journey

**From a 0.48 TF-IDF baseline to a 0.76 reranked submission**

---

This notebook describes the most important milestones of our project, academic citation retrieval. The challenge: given a query paper, rank 100 documents from a 20,000-paper corpus so that the papers it *actually cites* appear at the top. The scoring metric is **NDCG@10**.

We had two splits:
- **Train** (100 queries with ground-truth citations), used to iterate and measure progress locally.
- **Held-out** (100 queries, scored on Codabench), the final leaderboard metric.

Rather than listing every experiment (there were too many and some were not really relevant), this notebook keeps only the ideas that **shaped our final pipeline**, or that taught us something important when they *didn't* work. Each section explains:
1. The intuition behind the experiment.
2. What we actually did (with runnable or representative code).
3. What we learned and how it fed into the next step.

By the end, we will hopefully have explained the architecture of our best submission and why the simpler approaches we rejected were not enough.

## Roadmap of milestones

| # | Milestone | Train NDCG@10 | Why it mattered |
|---|-----------|---------------|-----------------|
| 1 | Data exploration | — | Understanding the dataset's structure & difficulty |
| 2 | TF-IDF baseline | 0.4841 | Establishes the floor for lexical retrieval |
| 3 | Dense baseline (MiniLM) | 0.5073 | Dense > sparse, but still weak |
| 4 | Stronger encoders (BGE-large) | ~0.68 | Model choice dominates early gains |
| 5 | Domain-finetuned BGE-small | ~0.70 | Fine-tuning on our corpus helps |
| 6 | Score-fusion pipeline (8 signals) | 0.7401 | Soft weighted sum — our first real competitive system |
| 7 | Hard-domain pipeline + cite re-ranking | 0.7552 | Emulating the competitor with a twist |
| 8 | Final reranker v7 (rr + bc + CE + soft) | **0.7603** | Combining marginal signals super-additively |

(The above scores are computed on the train set, the actual codabench score is slightly lower, often 0.01 worse)

The **final submission** sits at step 8. Everything else in this notebook is the trail of ideas that made it possible.

---
## 1. The Data

Before any model, we needed to understand what we were searching over. Three files matter:

- `queries.parquet` — 100 papers that each cite a few others.
- `corpus.parquet` — 20,000 candidate papers (includes the true citations plus a large pool of negatives).
- `qrels.json` — `{query_id: [cited_doc_id, ...]}` — the ground truth.

Each paper has a `title`, `abstract`, a `ta` concatenation, a `full_text` (5–100k chars), a `chunk_meta` describing section boundaries, and metadata (`domain`, `venue`, `year`).


In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

# Root of the info_retrieval project
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
sys.path.insert(0, str(ROOT))

from helpers import load_queries, load_corpus, load_qrels, evaluate, ndcg_at_k

queries = load_queries(DATA_DIR / 'queries.parquet')
corpus  = load_corpus(DATA_DIR / 'corpus.parquet')
qrels   = load_qrels(DATA_DIR / 'qrels.json')
query_domains = dict(zip(queries['doc_id'], queries['domain']))

print(f'Queries: {len(queries)}  |  Corpus: {len(corpus):,}  |  Qrels: {len(qrels)}')
print(f'Gold docs per query: mean={queries["n_relevant"].mean():.1f}, '
      f'min={queries["n_relevant"].min()}, max={queries["n_relevant"].max()}')

Queries: 100  |  Corpus: 20,000  |  Qrels: 100
Gold docs per query: mean=7.4, min=2, max=71


In [2]:
# Domain distribution is skewed: Biology and Medicine dominate, Philosophy has a single query.
# This asymmetry matters later as our hardest queries turned out to be in the rare domains.
print('Corpus domains (top 8):')
print(corpus['domain'].value_counts().head(8).to_string())
print()
print('Query domains (all):')
print(queries['domain'].value_counts().to_string())

Corpus domains (top 8):
domain
Biology                  4034
Medicine                 3662
Computer Science         3571
Chemistry                2461
Psychology               1306
Physics                  1271
Materials Science         932
Environmental Science     567

Query domains (all):
domain
Biology                  21
Medicine                 21
Computer Science         12
Chemistry                10
Psychology                6
Physics                   6
Materials Science         5
Environmental Science     3
Business                  2
Mathematics               2
Geography                 2
Engineering               2
Geology                   2
History                   1
Philosophy                1
Art                       1
Political Science         1
Sociology                 1
Economics                 1


**Key observations from exploration:**

1. Every gold document is in the corpus (100% coverage). Therefore, there is no *recall ceiling* from missing documents, any loss is a ranking failure, not a retrieval impossibility.
2. **97.6% of citations are within-domain.** A Biology paper overwhelmingly cites other Biology papers. This single fact drives our final architecture.
3. The distribution of `n_relevant` is heavy-tailed: most queries have 3–6 citations, some have 20+. NDCG@10 treats them fairly (normalised by ideal DCG).
4. Full texts are long (median ~10k chars), so representing a paper solely by its title+abstract discards a lot of signal.

---
## 2. Baseline: TF-IDF over Title + Abstract

We start with the simplest thing that could possibly work: bag-of-words cosine similarity over the title and abstract.

**Why start here?** Two reasons. First, it sets a floor every subsequent system must beat. Second, it reveals which queries are "easy" (word-overlap is enough) versus "hard" (paraphrasing, cross-domain terminology). Later on, error analysis on the baseline helped us decide which reranking signals were worth engineering.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def format_ta(row):
    return (str(row.get('title') or '').strip() + ' ' + str(row.get('abstract') or '').strip()).strip()

query_texts  = [format_ta(r) for _, r in queries.iterrows()]
corpus_texts = [format_ta(r) for _, r in corpus.iterrows()]

vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.95, stop_words='english')
C = vec.fit_transform(corpus_texts)
Q = vec.transform(query_texts)
sim = cosine_similarity(Q, C)

corpus_ids = corpus['doc_id'].tolist()
tfidf_sub = {qid: [corpus_ids[j] for j in np.argsort(-sim[i])[:100]]
             for i, qid in enumerate(queries['doc_id'])}

r = evaluate(tfidf_sub, qrels, ks=[10, 100], query_domains=query_domains, verbose=False)
print(f"TF-IDF NDCG@10 = {r['overall']['NDCG@10']:.4f}  |  Recall@100 = {r['overall']['Recall@100']:.4f}")

TF-IDF NDCG@10 = 0.4841  |  Recall@100 = 0.7512


**Result: NDCG@10 = 0.4841.**

Recall@100 is already 0.75 which means that TF-IDF *finds* most of the gold documents, it just ranks them poorly. This pattern (good recall, bad precision) held for every sparse method we tried and motivated the two-stage designs that came later.

> **Relevant scripts**
> - [`models/tfidf.py`](../models/tfidf.py) — stand-alone TF-IDF retriever used for the sparse baseline.
> - [`helpers.py`](../helpers.py) — `load_queries`, `load_corpus`, `load_qrels`, `evaluate`, metric functions (`ndcg_at_k`, `recall_at_k`, ...).

---
## 3. Dense Retrieval: From MiniLM to Domain-Finetuned BGE

As we saw during the course, sparse methods break when the query and cited paper use different vocabulary for the same concept (e.g., "neural network" vs "MLP"). Dense embeddings fix that by mapping semantically similar texts to nearby vectors.

We iterated through encoders in rough order of strength:

| Encoder | Dim | Train NDCG@10 (alone) |
|---------|-----|-----------------------|
| `all-MiniLM-L6-v2` (22M) | 384 | 0.5073 |
| `allenai/specter2_base` | 768 | ~0.41 alone, +weight helps little |
| `BAAI/bge-base-en-v1.5` | 768 | ~0.63 |
| `BAAI/bge-large-en-v1.5` | 1024 | **0.6776** |
| `intfloat/e5-large-v2` | 1024 | ~0.65 |
| Domain-finetuned BGE-small | 384 | **~0.70** |

The jump from MiniLM (0.51) to BGE-large (0.68) was the single biggest gain we got from any one change. Then we fine-tuned a BGE-small on (query, gold) pairs from our training qrels with hard negatives mined from BM25, a cheap GPU run that still matched BGE-large's quality at a third of the dimension.

### The generic recipe for every dense model
```
scores[i, j] = cosine(query_emb[i], corpus_emb[j])
ranked_per_query = argsort(-scores, axis=1)[:, :100]
```
All our dense embeddings are L2-normalised, so cosine similarity reduces to a plain dot-product.

In [4]:
# For instance here we use BGE-large as a stand-alone retriever on the training split.
from helpers import load_embeddings

BGE = DATA_DIR / 'embeddings' / 'BAAI_bge-large-en-v1.5'
cq_emb, cq_ids = load_embeddings(BGE / 'corpus_embeddings.npy', BGE / 'corpus_ids.json')
qq_emb, qq_ids = load_embeddings(BGE / 'train/query_embeddings.npy', BGE / 'train/query_ids.json')

# Align corpus to the canonical order used elsewhere
cid2i = {c: i for i, c in enumerate(cq_ids)}
C_al = cq_emb[[cid2i[c] for c in corpus_ids]]

qid2i = {q: i for i, q in enumerate(qq_ids)}
Q_al = np.stack([qq_emb[qid2i[q]] for q in queries['doc_id']])

sim = Q_al @ C_al.T
bge_sub = {qid: [corpus_ids[j] for j in np.argsort(-sim[i])[:100]]
           for i, qid in enumerate(queries['doc_id'])}

r = evaluate(bge_sub, qrels, ks=[10, 100], query_domains=query_domains, verbose=False)
print(f"BGE-large alone   NDCG@10 = {r['overall']['NDCG@10']:.4f}")

BGE-large alone   NDCG@10 = 0.5580


**Lesson:** any single encoder, even the best available, scores well below what the competition demanded. To push further, we had to combine multiple signals. That is the entire purpose of the next section.

> **Relevant scripts**
> - [`scripts/embed.py`](../scripts/embed.py) : generic sentence-transformers embedding driver. Used to pre-compute corpus + query vectors for every encoder we tried (MiniLM, BGE-base/large, E5, Specter2, SciNCL, GTE-ModernBERT).
> - [`scripts/finetune_biencoder.py`](../scripts/finetune_biencoder.py) : fine-tunes BGE-small on our (query, gold) pairs with BM25-mined hard negatives.
> - [`models/dense.py`](../models/dense.py) standalone dense retriever: loads a cached embedding pair and ranks by cosine.
> - [`models/bm25.py`](../models/bm25.py) + [`scripts/precompute_bm25.py`](../scripts/precompute_bm25.py) : BM25 baseline and its pre-computation of the score matrix.

---
## 4. Score-Fusion Pipeline (NDCG@10 = 0.7401)

Different retrievers make different mistakes. For instance BGE captures paraphrase where BM25 captures rare technical terms. A domain-specific encoder like SPECTER2 understands scientific writing conventions. A *citation-context* signal captures what the query paper actually says about what it cites.

Our first competitive system was a **weighted sum of min-max-normalised signal scores**. Each signal produces a length-20,000 (one dim per document of the corpus)score vector per query, then we normalise each to [0, 1] and add them with tuned weights.

The eight signals that survived ablation:

| Signal | Weight | What it captures |
|--------|--------|------------------|
| Finetuned BGE-small | 0.50 | Domain-adapted semantic similarity |
| BGE-large | 0.50 | General semantic similarity |
| BM25 (title+abstract) | 0.35 | Rare-term lexical overlap |
| Same-domain mask | 0.38 | Binary indicator — query & doc in same field |
| Pooled cite-context vs corpus | 0.15 | Mean-pooled embedding of citation sentences |
| Per-sentence cite-context (BGE) | 0.95 | Max-sim between each cite sentence and candidate |
| Per-sentence cite-context (E5) | 0.35 | Same idea, different encoder |
| Cite-sentence vs corpus body chunks | 0.07 | Chunk-level match against full text |

### Why the cite-context signals dominate

A citation sentence is an extremely specific query, it literally describes *what the paper being cited is about*. Encoding those sentences and matching them against corpus documents gave us our largest single-signal contribution (weight 0.95). We pre-extracted them with a regex that finds bracketed or parenthesised citations (`[12]`, `(Smith et al., 2020)`) and embedded each sentence individually.

### Why the domain mask works

Because 97.6% of gold citations share the query's domain, a tiny additive bonus on same-domain candidates nudges dozens of correct papers up by a handful of ranks, which is enough to show up in NDCG@10.

### Formal recipe (score space), mm refers to min_max normalization
```
score(q, d) =   0.50 · mm(ft_small(q) · ft_small(d))
              + 0.50 · mm(bge_large(q) · bge_large(d))
              + 0.35 · mm(BM25(q, d))^1.2
              + 0.38 · 1{domain(q) = domain(d)}
              + 0.15 · mm(pooled_cite(q) · bge_large(d))
              + 0.95 · mm(max_i bge_cite_i(q) · bge_large(d))
              + 0.35 · mm(max_i e5_cite_i(q)  · e5_large(d))
              + 0.07 · mm(max_i bge_cite_i(q) · corpus_chunk(d))
```
The exponent 1.2 on BM25 was tuned by grid search. It appears that BM25 scores are long-tailed, and raising them to a slightly higher power than 1 separates strong matches from mediocre ones.

**Train NDCG@10 = 0.7401**, which was our first score that felt really competitive. Extensive coordinate ascent and 5-fold cross-validation over these weights confirmed this was a **local optimum**.

### What we tried and rejected in this phase

Several other candidate signals were trained and measured against this pipeline. None of them improved it:

- **Extra encoders as standalone signals** (Specter2, SciNCL, E5-base, GTE-ModernBERT): all redundant with BGE-large or strictly weaker on train.
- **Pseudo-relevance feedback, venue matching, TF-IDF variants**: flat or negative deltas compared to this pipeline.
- **Reciprocal-rank fusion of multiple configurations**: RRF didn't beat the score-space weighted sum in our setup.
- **Full-text BM25 / title-only BM25**: title+abstract strictly dominated.

These null results were instructive: they told us we had saturated the "just add another signal" axis. The next improvement had to come from a **different architecture**.

> **Relevant scripts**
> - [`models/pipeline.py`](../models/pipeline.py) — the 8-signal score-fusion pipeline (the 0.7401 model); loads all cached signals and combines them with the weights listed above.
> - [`models/score_fusion.py`](../models/score_fusion.py) — generic score-space fusion utilities (min-max normalisation, weighted sum).
> - [`models/rrf.py`](../models/rrf.py) — the RRF alternative we tried and rejected.
> - [`models/cite_context.py`](../models/cite_context.py) + [`scripts/embed_cite_contexts.py`](../scripts/embed_cite_contexts.py) — citation-sentence extraction and embedding (pooled + per-sentence, both BGE and E5).
> - [`models/fullchunk.py`](../models/fullchunk.py) + [`scripts/embed_corpus_chunks.py`](../scripts/embed_corpus_chunks.py) — corpus body-chunk embeddings used for the chunk-cite signal.
> - [`tuning/cv_weights_fast.py`](../tuning/cv_weights_fast.py) — 5-fold CV grid search over the eight signal weights.
> - [`tuning/tune_bm25.py`](../tuning/tune_bm25.py) — BM25 `k1`/`b` and score-power (`^1.2`) grid search.

---
## 5. The Hard-Domain Pivot (NDCG@10 = 0.7552)

When listening at the presentations last Thursday, we noticed that the top competitor sat around 0.7489 on held-out. We tried to implement some of their best ideas to improve our own model and ended up with a very different architecture:

**Hard-domain filtering**: instead of *softly* boosting same-domain candidates with a +0.38 mask (which was what we did until now), *only* rank documents from the query's domain. Then fall back to global results for the last few slots.

Combined with a long-context encoder (`Alibaba-NLP/gte-modernbert-base` at 8192 tokens on full text), this gives a much cleaner candidate set. Our version uses 7 signals inside the hard-domain filter:

Here is a simple implementation in pseudo code of what we actually do:
```
for each query q:
    dom_candidates = { d in corpus : domain(d) == domain(q) }
    score_dom = Σᵢ wᵢ · mm(signalᵢ(q, d))   for d in dom_candidates
    top_dom = top-k of dom_candidates by score_dom
    if |top_dom| < cand_n:
        fill the remainder from the global top-ranked docs
```

With the right weights (found by 5000-sample Dirichlet search), the 7-signal hard pipeline alone reaches **0.7447** on train, already beating our soft pipeline. This was Stage 1.

### The winning twist: add cite re-ranking on top

The hard pipeline gives us a strong candidate ordering but ignores the citation context, which was our soft pipeline's strongest signal. So we kept the 7-signal ranking as a **base** and re-scored its top-100 candidates with our per-sentence BGE cite-context similarity:

```
base(q, d_rank)  = 1 / (rank + 1)                        # reciprocal rank of d in 7-signal list
cite(q, d)       = mm( max_i  bge_cite_i(q) · bge_large(d) )
final(q, d)      = base + 0.22 · cite
```

**Crucial implementation detail**: if you add raw cosine values instead of min-max-normalised ones, cite scores in the 0.5–0.9 range overwhelm the reciprocal-rank base (max 1.0, then 1/2, 1/3…) and the whole thing collapses to 0.63, as we realized when we first implemented it. mm-normalisation inside each query's candidate pool is crucial here.

**Train NDCG@10 = 0.7552**, a +0.015 jump over the 7-signal base, and crucially *not* a redundant signal: the hard-domain filter and cite re-ranker capture different information, effectively improving our score.

> **Relevant scripts**
> - [`models/hard_domain_retrieval.py`](../models/hard_domain_retrieval.py) — the 7-signal hard-domain pipeline (Stage 1) with Dirichlet weight search (reaches 0.7447 alone).
> - [`scripts/embed_gte_modernbert.py`](../scripts/embed_gte_modernbert.py) + [`scripts/embed_gte_ft_8k_gpu.py`](../scripts/embed_gte_ft_8k_gpu.py) — GTE-ModernBERT full-text embedding at 8192 tokens (the dominant signal of Stage 1, ~2h45 GPU run).
> - [`scripts/embed_gte_large_chunks.py`](../scripts/embed_gte_large_chunks.py) — body-chunk embeddings with `thenlper/gte-large`, used for the cMaxSim signal.
> - [`models/hard_pipeline_with_cite.py`](../models/hard_pipeline_with_cite.py) — Stage 1 + the baseline cite re-ranker (reaches **0.7552**, `w_bc=0.22`, `dom_k=80`, `cand_n=100`).

---
## 6. Error Analysis: Where Do We Still Lose Points?

At 0.7552, improvements became harder to find by trying new signals. We shifted to *diagnosing failures*.

For each training query we computed NDCG@10 and inspected the oracle ceiling: if we could **perfectly reorder** the current top-100 candidates, what would we score?

In [5]:
# Illustrative oracle analysis using the TF-IDF candidates from earlier.
# In practice we ran this against the 0.7552 candidate pool.
def oracle_ndcg(candidates_per_query, qrels):
    scores = []
    for qid, cands in candidates_per_query.items():
        relevant = set(qrels.get(qid, []))
        if not relevant:
            continue
        # Perfect reorder: put every relevant doc at the top.
        reordered = [d for d in cands if d in relevant] + [d for d in cands if d not in relevant]
        scores.append(ndcg_at_k(reordered, relevant, 10))
    return float(np.mean(scores))

print(f"TF-IDF actual NDCG@10 : {evaluate(tfidf_sub, qrels, verbose=False)['overall']['NDCG@10']:.4f}")
print(f"TF-IDF oracle  NDCG@10 : {oracle_ndcg(tfidf_sub, qrels):.4f}")
print('(Gap = headroom a perfect reranker could capture)')

TF-IDF actual NDCG@10 : 0.4841
TF-IDF oracle  NDCG@10 : 0.8393
(Gap = headroom a perfect reranker could capture)


On our 0.7552 candidate pool, the oracle reached **0.9506**, meaning a perfect reranker of the *existing* top-100 would gain us +0.195 NDCG@10. Conclusion : The candidate set is fine, **the ordering is the part causing the score to drop.**

Other findings from the error analysis:
- **66 / 100** queries already have *all* their gold docs in the top-100.
- The worst domains are Philosophy (0.22), Business (0.55), and Computer Science (0.60), in other words the rare domains with few in-domain candidates.
- The hardest individual queries typically have a single relevant doc sitting at rank 3–19: *just outside* the top-10, where NDCG penalises heavily.

This pointed to one direction: **a better reranker of the existing top-100**, not more retrieval signals.

> **Relevant scripts**
> - [`scripts/error_analysis.py`](../scripts/error_analysis.py) — per-query NDCG breakdown, oracle-reorder headroom, worst-domain and worst-query diagnostics.
> - [`scripts/deep_error_analysis.py`](../scripts/deep_error_analysis.py) — the earlier version used during the soft-pipeline era to identify weak domains.

---
## 7. The Reranker Breakthrough (NDCG@10 = 0.7603)

We had three weakly-helpful signals that each moved the score by roughly +0.001 in isolation:

- A **cross-encoder** (`BAAI/bge-reranker-base`) scoring each (query, candidate) pair end-to-end (+0.0009).
- Our **soft-pipeline score** from step 4 used as a feature on the hard-pipeline candidates (+0.0011).
- Tuning the **power of the reciprocal-rank base** `rr^p` (+0.0003).

Each was marginal alone. We thought first about discarding them. But error analysis told us we only needed *ordering* information, and each of these looks at the candidates through a different lens.

We ran a **joint 4-D grid search** over `(rr_pow, w_bc, w_ce, w_soft)` on top of the hard-domain candidates:

```
score(q, d) = rr(q, d)^p
            + w_bc   · mm(cite_context_sim(q, d))
            + w_ce   · mm(bge_reranker_base(q, d))
            + w_soft · mm(soft_pipeline_score(q, d))
```

where `mm(·)` is per-query min-max normalisation over the 100 candidates. The best configuration found:

| Parameter | Value |
|-----------|-------|
| `rr_pow`  | 1.00 |
| `w_bc`    | 0.44 |
| `w_ce`    | 0.04 |
| `w_soft`  | 0.25 |

**Train NDCG@10 = 0.7603** — an additional +0.0051 on top of the 0.7552 baseline.

### Why does this work when the parts look so weak?

Look at the individual deltas from the baseline (`w_bc=0.22`, everything else zero):

- Adding CE alone: +0.001
- Adding soft alone: +0.001
- Tuning `rr_pow`: +0.0003

But the baseline itself *under-weighted* `bc`, the old value of 0.22 was set when CE and soft weren't in the mix. When we re-tuned `bc` jointly to 0.44 and added small positive weights on CE and soft, the four features combined **super-additively**, unlocking a broad plateau around 0.760. The plateau seems hard to break, nearby configurations like `(0.9, 0.48, 0.04, 0.25)` and `(1.0, 0.38, 0.04, 0.28)` all score ~0.760.

### Cross-validation sanity check

We validated the result with 5-fold CV (20 queries held out per fold, deterministic seed):
- Using the *global* config on each held-out fold: mean NDCG@10 = **0.7603**.
- Re-tuning per fold (train on 80, test on 20): mean = **0.7563**, worse.



> **Relevant scripts**
> - [`scripts/crossencoder_rerank_v2.py`](../scripts/crossencoder_rerank_v2.py) : cross-encoder reranking with `BAAI/bge-reranker-base`. CE scores are cached in `data/ce_cache/`.
> - [`scripts/compute_soft_scores.py`](../scripts/compute_soft_scores.py) : computes and caches the soft-pipeline score feature for every (query, candidate) pair (cached in `data/soft_scores/`).
> - [`models/reranker_v7.py`](../models/reranker_v7.py) : **the 4-D joint grid search** over `(rr_pow, w_bc, w_ce, w_soft)` that found the 0.7603 optimum.
> - [`tuning/cv_rerank_v7.py`](../tuning/cv_rerank_v7.py) : 5-fold cross-validation confirming the gain is not overfitting.

---
## 8. The Final Pipeline — End to End

Pulling every step together, here is the full architecture of our best submission. All the heavy computation (embeddings, BM25 scores, cross-encoder scores, soft-pipeline scores) is **pre-computed and cached**; the reranker itself is pure NumPy and takes a few seconds per run.

```
┌─────────────────────────────────────────────────────────────────────┐
│  STAGE 1 — Hard-domain 7-signal retrieval                           │
│  ---------------------------------------------------------------    │
│  Signals (with their Dirichlet-optimised weights):                  │
│    • MiniLM cosine              0.074                               │
│    • BGE-large cosine           0.000                               │
│    • BM25 body (reciprocal rank) 0.070                              │
│    • TF-IDF TA                  0.000                               │
│    • TF-IDF FT (10k chars)      0.160                               │
│    • GTE-ModernBERT FT @ 8192   0.506  (dominant signal)            │
│    • GTE-large chunk max-sim    0.190                               │
│                                                                     │
│  Restrict to same-domain candidates; fall back to global if < 100.  │
│  Output: top 100 candidates per query (dom_k=80, cand_n=100).       │
└─────────────────────────────────────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│  STAGE 2 — Four-feature reranker                                    │
│  ---------------------------------------------------------------    │
│  For each of the 100 candidates, compute:                           │
│    rr    = 1 / (rank + 1)           — base order from Stage 1       │
│    bc    = mm(max_i bge_cite_i · bge_doc)                           │
│    ce    = mm(bge-reranker-base(query, doc))                        │
│    soft  = mm(soft_pipeline_score)                                  │
│                                                                     │
│  Final score:  rr^1.0 + 0.44·bc + 0.04·ce + 0.25·soft               │
│  Return top-100 of the reranked order.                              │
└─────────────────────────────────────────────────────────────────────┘
```

### How the code is organised

```
info_retrieval/
├── helpers.py                                loaders, metric functions
├── data/embeddings/                          pre-computed encoders (MiniLM, BGE-*, E5, GTE-MB, ...)
├── data/bm25_cache/                          cached BM25 score matrices
├── data/ce_cache/                            cached cross-encoder scores (train + held-out)
├── data/soft_scores/                         cached soft-pipeline scores
├── models/
│   ├── pipeline.py                           soft 8-signal pipeline (0.7401)
│   ├── hard_domain_retrieval.py              7-signal hard-domain pipeline (0.7447–0.7495)
│   ├── hard_pipeline_with_cite.py            Stage 1 + cite re-rank (0.7552)
│   └── reranker_v7.py                        4-D grid search that found the 0.7603 optimum
│                                             (also generates held-out submissions)
├── scripts/
│   ├── crossencoder_rerank_v2.py             cross-encoder reranking + CE cache builder
│   └── compute_soft_scores.py                computes + caches the soft-pipeline feature
├── tuning/
│   └── cv_rerank_v7.py                       5-fold cross-validation
└── submissions/
    └── reranker_v7_held_out.json             final Codabench submission
```

### Reproducing the final score

From the project root (assuming all caches are present):

```bash
# Train eval — reproduces 0.7603
python models/reranker_v7.py --split train

# Generate the held-out submission (CE cache is auto-built if missing)
python models/reranker_v7.py --split held_out
```

> **Relevant scripts (end-to-end reproduction)**
> - Stage 1 retrieval: [`models/hard_pipeline_with_cite.py`](../models/hard_pipeline_with_cite.py) (loads all signals + provides `load_signals_plus_cite` / `retrieve`).
> - Stage 2 reranker: [`models/reranker_v7.py`](../models/reranker_v7.py) — handles both train eval (`--split train`) and held-out submission (`--split held_out`); auto-builds the CE cache if missing.
> - Pre-computed caches consumed by the pipeline: `data/embeddings/*`, `data/bm25_cache/*`, `data/ce_cache/ce_scores_{train,held_out}_BAAI_bge-reranker-base_ta.npz`, `data/soft_scores/soft_scores_{train,held_out}.npz`.

In [6]:
# Here is the 10-line heart of the final reranker,
# so the combination formula is visible in one place.

def rerank_v7(rr, bc, ce, soft, w_bc=0.44, w_ce=0.04, w_soft=0.25, rr_pow=1.0):
    """Rerank 100 candidates given their four feature vectors (each length 100).

    rr    : reciprocal-rank from Stage 1 (already in [0, 1])
    bc    : cite-context similarity, raw
    ce    : cross-encoder score, raw
    soft  : soft-pipeline score, raw

    Returns the index order that sorts candidates best-first.
    """
    def mm(x):
        x = np.asarray(x, np.float32)
        lo, hi = x.min(), x.max()
        return np.zeros_like(x) if hi == lo else (x - lo) / (hi - lo + 1e-9)

    score = rr**rr_pow + w_bc * mm(bc) + w_ce * mm(ce) + w_soft * mm(soft)
    return np.argsort(-score)

# Sanity check that the formula runs on fake data
rng = np.random.default_rng(0)
demo = rerank_v7(
    rr   = 1.0 / (np.arange(100) + 1),
    bc   = rng.random(100),
    ce   = rng.random(100),
    soft = rng.random(100),
)
print(f'Reranker produced an ordering of {len(demo)} candidates.')
print(f'First 5 indices: {demo[:5].tolist()}')

Reranker produced an ordering of 100 candidates.
First 5 indices: [0, 5, 9, 78, 4]


---
## 9. Takeaways

Looking back at the full trajectory, four lessons stand out:

1. **Model choice dominates everything else early on.** TF-IDF → MiniLM → BGE-large → domain-finetuned BGE-small was a +0.2 jump in NDCG@10 at almost no engineering cost. Before trying anything clever, pick strong encoders.

2. **Citation context is gold.** Explicitly extracting the sentences in the query paper that *refer to* citations, embedding each one, and taking a max over matches against each candidate was our single most valuable engineered feature, in both the soft and hard pipelines.

3. **Marginal signals are not worthless.** Three features that each contributed +0.001 in isolation combined super-additively to +0.005 when the other weights were re-tuned jointly. We should never drop a feature because it "only" adds one point but rather check it in combination.

4. **Error analysis greatly helps** Once we computed the oracle ceiling on the candidate pool (0.9506 vs actual 0.7552), we knew any further progress had to come from reranking, not retrieval. That one diagnostic redirected the entire final phase of the project.

### Current best scores

| Split | NDCG@10 |
|-------|---------|
| Train | **0.7603** |
| Held-out (Codabench) | 0.749 |

